# 03 · Feature study: correlations, lead–lag, redundancy and selection

> **Public copy (NDA).** Outputs are cleared and private values are removed from the text. Run it locally with the NDA data in `data/`; never push it with outputs. Figures are not saved from this copy (`save` is a no-op): the public figures come from `public/reports/make_public_figures.py`.


**Why this notebook.** The kick-off deck scores feature engineering on one rule: *"enrich with public drivers; prove each feature was known at forecast time"* (slide 16, item 3). Notebook 02 showed that the drivers matter more than the model: market prices lead the Eurostat PPIs, nickel leads the stainless surcharge, Brent leads plastics. This notebook studies the candidate features in a systematic way, then writes a **training-ready, leakage-safe feature set**.

| § | Question | Output |
|---|---|---|
| 1 | What do we have? | target and driver universe |
| 2 | Levels or returns? | ADF table → work on Δlog |
| 3 | How do the series co-move? | correlation matrices (levels, Δlog Pearson/Spearman, family blocks, rolling) |
| 4 | Which series are redundant? | dedup table (|r| > 0.95 on monthly changes) |
| 5 | Who leads whom? | lead–lag CCF heatmaps, Granger tests |
| 6 | Which engineered features predict the 3/6/9-month return? | Spearman ρ, mutual information, regime stability |
| 7 | Are the kept features collinear? | VIF pruning |
| 8 | Is anything leaking? | `check_no_leakage` at h = 3/6/9 |
| 9 | What do we train on? | `feature_selection.parquet`, `features_h{3,6,9}.parquet` |

Convention (same as notebook 02): at origin *t*, the target for horizon *h* is `log(y[t+h] / y[t])`. Every driver is shifted by `max(0, lag_driver − lag_target)` publication months (`src/features.py::_shift_for`).

In [ ]:
import sys, warnings, io, contextlib
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl, seaborn as sns
from scipy.cluster import hierarchy
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import mutual_info_regression
import src.features as fe, src.preprocessing as pp

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
mpl.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.6,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.prop_cycle": mpl.cycler(color=PAL), "lines.linewidth": 1.6, "legend.frameon": False,
    "legend.fontsize": 8, "font.family": "sans-serif", "figure.dpi": 110,
})
DIV = mpl.colors.LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])  # blue < 0 < red
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / "data" / "processed"
save = lambda fig, name: None  # public copy: never write figures
HORIZONS = (3, 6, 9)
REGIMES = {"pre-2021": ("2012-01-01", "2020-12-01"), "2021-22 shock": ("2021-01-01", "2022-12-01"),
           "2023+": ("2023-01-01", "2026-10-01")}

## 1 · Data

- `panel_monthly.parquet` is the **un-imputed** panel. We use it for every statistic, so forward-filled ragged-edge months do not create fake zero returns.
- `series_catalog.parquet` gives each series its role, family, BOM material and publication lag.
- `fe.build_panel()` returns the 12 forecasting targets used in notebook 02. We add **EPOXY_RESIN** as a 13th study target, because the deck lists insulating materials and epoxy is the only priced one.

In [ ]:
raw = pd.read_parquet(PROC / "panel_monthly.parquet")
cat = pd.read_parquet(PROC / "series_catalog.parquet")
T, X = fe.build_panel()
TARGETS = list(T.columns)
STUDY_TARGETS = TARGETS + ["EPOXY_RESIN"]
RENAME = {"SS304_ALLOY_DE": "CRU_SS304_DE_ALLOY"}          # build_panel name -> processed-panel name
P = raw.rename(columns={v: k for k, v in RENAME.items()})   # panel with build_panel names for the targets
print("panel:", raw.shape, "| targets:", T.shape, "| notebook-02 drivers:", X.shape)
cat.groupby(["family", "role"]).size().unstack(fill_value=0)

## 2 · Levels or returns?

ADF test on log-levels and on Δlog. If levels are I(1) and Δlog is stationary, correlations between **levels** are spurious, because two trending series always look correlated. Every later step therefore works on **monthly log-changes** or on **forward h-month log-returns**.

In [ ]:
def dlog(df: pd.DataFrame) -> pd.DataFrame:
    # log-change for strictly positive series; plain difference for spreads that can be <= 0
    pos = (df > 0).all() | df.isna().all()
    out = np.log(df.loc[:, pos]).diff()
    return pd.concat([out, df.loc[:, ~pos].diff()], axis=1)[df.columns]

SHORTLIST = ["S03", "S04", "S05", "S06", "S08", "D01", "D02", "ED01", "ED02", "E09", "E13", "E22",
             "LME_CU_SPREAD_REL", "LME_AL_SPREAD_REL", "LME_CU_STOCKS", "LME_AL_STOCKS",
             "EPOXY_ECH", "EPOXY_BENZENE", "EPOXY_ACETONE"]
SHORTLIST = [s for s in SHORTLIST if s not in STUDY_TARGETS]
D = dlog(P)

def adf_p(s):
    s = s.dropna()
    return adfuller(s, autolag="AIC")[1] if len(s) > 30 else np.nan

rows = []
for s in STUDY_TARGETS + SHORTLIST:
    lv = np.log(P[s]) if (P[s].dropna() > 0).all() else P[s]
    rows.append({"series": s, "role": "target" if s in STUDY_TARGETS else "driver", "n": P[s].notna().sum(),
                 "ADF p (level)": adf_p(lv), "ADF p (Δ)": adf_p(D[s])})
adf = pd.DataFrame(rows).set_index("series")
adf["verdict"] = np.where((adf["ADF p (level)"] > 0.05) & (adf["ADF p (Δ)"] < 0.05), "I(1) → use Δlog",
                  np.where(adf["ADF p (level)"] <= 0.05, "level stationary", "check"))
adf

**Reading.** Every price is I(1), as notebook 01 found. D01's Δlog p-value (≈ 0.09) reflects the 2025-26 copper trend, not a unit root in returns. The exceptions are the LME **curve and stocks**: the Al spread and Cu stocks are level-stationary, and the Cu spread is borderline (p ≈ 0.06). These are mean-reverting state variables, so they enter as levels or deviations from their 24-month mean (`fe.LEVEL_FEATURES`), not as returns.

## 3 · Correlation matrices

### 3a · Levels (for reference only: spurious)

In [ ]:
lvl_cols = STUDY_TARGETS + SHORTLIST
C_lvl = np.log(P[lvl_cols].where(P[lvl_cols] > 0)).corr(min_periods=36)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(C_lvl, cmap=DIV, center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={"shrink": .6}, xticklabels=True, yticklabels=True)
ax.set_title("Pearson correlation of log-LEVELS: inflated by common trends; do not use for selection")
save(fig, "17_corr_levels"); plt.show()

### 3b · Monthly log-changes over a de-duplicated universe

First we prune redundant series (§4 explains the rule) so the matrix stays readable. Then we show Pearson and Spearman, ordered by hierarchical clustering.

In [ ]:
# --- redundancy pruning (used here and detailed in §4) ---
DRIVER_SET = set(sum(fe.DRIVERS.values(), [])) | set(fe.COMMON_DRIVERS) | set(SHORTLIST)
def priority(s: str) -> tuple:
    name = RENAME.get(s, s)
    if s in STUDY_TARGETS: p = 0
    elif s in DRIVER_SET: p = 1
    elif name.endswith(("_DE", "_FR")) and name.startswith("CRU_"): p = 2
    elif name.startswith(("MEPS_", "LME_CU_CASH", "LME_CU_3M", "LME_AL_CASH", "LME_AL_3M", "S01", "S02")): p = 5
    elif name.startswith("CRU_LONG"): p = 4
    else: p = 3
    return (p, -P[s].notna().sum(), s)

obs = D.notna().sum()
univ = [c for c in sorted(D.columns, key=priority) if obs[c] >= 60 and D[c].std() > 0]
C_all = D[univ].corr(min_periods=36)
kept, dropped, twins = [], [], []
for s in univ:
    dup = [k for k in kept if abs(C_all.loc[s, k]) > 0.95]
    if dup and s in STUDY_TARGETS:          # targets are never dropped, only flagged
        twins.append({"target": s, "near_duplicate_of": dup[0], "r_dlog": C_all.loc[s, dup[0]]})
    (dropped.append({"dropped": s, "kept_instead": dup[0], "r_dlog": C_all.loc[s, dup[0]]})
     if dup and s not in STUDY_TARGETS else kept.append(s))
dedup = pd.DataFrame(dropped)
print(f"{len(univ)} series with ≥60 monthly changes → {len(kept)} kept, {len(dropped)} redundant")
pd.DataFrame(twins)

In [ ]:
def clustered_heatmap(C, title, name, size=(13, 11)):
    Z = hierarchy.linkage(1 - C.fillna(0).abs().values[np.triu_indices(len(C), 1)], "average")
    order = C.index[hierarchy.leaves_list(Z)]
    fig, ax = plt.subplots(figsize=size)
    sns.heatmap(C.loc[order, order], cmap=DIV, center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={"shrink": .6},
                xticklabels=True, yticklabels=True)
    ax.tick_params(labelsize=6.5); ax.set_title(title)
    save(fig, name); plt.show()
    return order

C_p = D[kept].corr(method="pearson", min_periods=36)
C_s = D[kept].corr(method="spearman", min_periods=36)
order = clustered_heatmap(C_p, "Pearson correlation of monthly Δlog (de-duplicated universe, clustered)", "18_corr_dlog_pearson")
_ = clustered_heatmap(C_s, "Spearman correlation of monthly Δlog (robust to the 2021-22 outliers)", "18b_corr_dlog_spearman")
focus = [c for c in STUDY_TARGETS + SHORTLIST if c in kept]
gap = (C_p - C_s).loc[focus, focus].stack()
gap = gap[gap.index.get_level_values(0) < gap.index.get_level_values(1)]
print("Largest Pearson − Spearman gaps among targets/short-listed drivers (a few extreme months drive Pearson):")
gap.reindex(gap.abs().sort_values(ascending=False).index).head(10).rename("pearson − spearman")

### 3c · Family blocks (the relations a buyer cares about)

In [ ]:
BLOCKS = {
    "Steel (carbon / galvanised)": ["CRU_CR_DE", "CRU_CR_FR", "CRU_HDG_DE", "S09", "S10", "S04", "S05", "ED02", "E09", "S06"],
    "Stainless & non-ferrous": ["CRU_SS304_DE", "SS304_ALLOY_DE", "S03", "D01", "D02", "S08", "S06",
                                "LME_CU_SPREAD_REL", "LME_AL_SPREAD_REL", "LME_CU_STOCKS", "LME_AL_STOCKS"],
    "Energy → plastics & epoxy": ["S11", "EPOXY_RESIN", "EPOXY_ECH", "EPOXY_BENZENE", "EPOXY_ACETONE", "ED01", "ED02", "E13", "E22", "S06"],
}
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
for ax, (title, cols) in zip(axes, BLOCKS.items()):
    cols = [c for c in cols if c in D]
    sns.heatmap(D[cols].corr(min_periods=36), cmap=DIV, center=0, vmin=-1, vmax=1, annot=True, fmt=".2f",
                annot_kws={"size": 6.5}, ax=ax, cbar=False, xticklabels=True, yticklabels=True)
    ax.set_title(title); ax.tick_params(labelsize=7)
fig.suptitle("Contemporaneous correlation of monthly Δlog by family", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); save(fig, "19_corr_family_blocks"); plt.show()

### 3d · Rolling 24-month correlation: are the relations stable?

In [ ]:
PAIRS = [("S09", "CRU_CR_DE", 1), ("SS304_ALLOY_DE", "S03", 1), ("S11", "ED01", 1),
         ("EPOXY_RESIN", "EPOXY_ECH", 0), ("D01", "S06", 0), ("CRU_HDG_DE", "S04", 1)]
fig, ax = plt.subplots(figsize=(11, 4))
for (a, b, k), c in zip(PAIRS, PAL):
    ax.plot(D[a].rolling(24, min_periods=18).corr(D[b].shift(k)), color=c, label=f"{a} vs {b} (lag {k})")
ax.axhline(0, color=AXIS, lw=0.8); ax.set_ylim(-1, 1); ax.legend(ncol=3, loc="lower left")
ax.set_title("Rolling 24-month correlation of Δlog (driver lagged by k months)")
save(fig, "20_rolling_corr"); plt.show()

**Reading.**
- **Pass-through links are stable.** CRU coil → steel PPI (lag 1), nickel → SS304 alloy surcharge (lag 1) and Brent → plastics PPI (lag 1) stay positive in every 24-month window. CRU → S09 never drops below 0.27 and is usually above 0.45. These are safe features.
- **Iron ore → HDG coil is regime-dependent.** It is strong before 2020 and in 2023-25, and close to zero in 2020 and 2026.
- **ECH → epoxy resin is weak on monthly changes (r ≈ 0.38).** Benzene is stronger (0.66). The resin moves on contract lags, so the cost-recipe link holds in levels more than in month-on-month changes.
- **Copper vs EUR/USD changes sign (−0.4 to +0.5).** Use EUR/USD as a currency conversion, not as a predictive signal.

## 4 · Redundancy table

Rule: walk the series in priority order (targets → known drivers → CRU DE/FR → others → long products → MEPS / raw LME duplicates). A series is dropped if its monthly Δlog correlates above 0.95 with a series already kept. The kept representative is the one that matches the BOM and part-mapping language (CRU DE/FR, D01/D02).

In [ ]:
print("Near-duplicate pairs by % change (preprocessing.near_duplicate_columns, thresh 0.95):",
      len(pp.near_duplicate_columns(raw, thresh=0.95)))
dedup.groupby("kept_instead").agg(n_dropped=("dropped", "size"), dropped=("dropped", lambda s: ", ".join(s[:6]) + (" …" if len(s) > 6 else ""))) \
     .sort_values("n_dropped", ascending=False)

## 5 · Lead–lag

### 5a · Structural lead: which driver's change *today* shows up in the target's change *k months later*?

CCF of monthly Δlog, driver at *t − k* against target at *t*, for k = 0…6. This is the raw economic lag, before publication delays. It should reproduce the known relations: CRU CR → S09 at lag 1 ≈ 0.7, nickel → SS304 surcharge at lag 1 ≈ 0.66, Brent → plastics at lag 1 ≈ 0.6.

In [ ]:
LL_DRIVERS = [c for c in dict.fromkeys(SHORTLIST + ["CRU_CR_DE", "CRU_HDG_DE", "S09", "S10", "D01", "S03", "S04"]) if c in D]
def ccf_table(tg, drivers, lags=range(0, 7)):
    res = {}
    for d in drivers:
        if d == tg: continue
        r = pd.Series({k: D[tg].corr(D[d].shift(k), min_periods=36) for k in lags})
        res[d] = {"best_lag": int(r.abs().idxmax()), "r_best": r.loc[r.abs().idxmax()], "r_lag0": r.loc[0]}
    return pd.DataFrame(res).T

cc = {tg: ccf_table(tg, LL_DRIVERS) for tg in STUDY_TARGETS}
R = pd.DataFrame({tg: t.r_best for tg, t in cc.items()}).T.astype(float)
L = pd.DataFrame({tg: t.best_lag for tg, t in cc.items()}).T
annot = R.round(2).astype(str) + "\n(" + L.fillna(-1).astype(int).astype(str) + ")"
fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(R, cmap=DIV, center=0, vmin=-0.8, vmax=0.8, annot=annot.where(R.abs() >= 0.3, ""), fmt="",
            annot_kws={"size": 6.5}, ax=ax, cbar_kws={"shrink": .6}, xticklabels=True, yticklabels=True)
ax.set_title("Best |corr| of Δlog driver(t−k) vs Δlog target(t), k = 0–6 (best lag in brackets, |r| ≥ 0.3 annotated)")
ax.set_xlabel("driver"); ax.set_ylabel("target")
save(fig, "21_leadlag_ccf"); plt.show()

check = [("S09", "CRU_CR_DE"), ("SS304_ALLOY_DE", "S03"), ("S11", "ED01"), ("CRU_CR_DE", "S04")]
pd.DataFrame([{"target": t, "driver": d, **cc[t].loc[d].to_dict()} for t, d in check])

### 5b · Granger causality on the strongest pairs

The F-test asks whether the driver's past Δlog improves an AR model of the target's Δlog (max lag 3). With about 100–170 months, treat p < 0.05 as "useful", not as "causal".

In [ ]:
top = (R.abs().stack().rename("r").reset_index().rename(columns={"level_0": "target", "level_1": "driver"})
       .query("r >= 0.3").sort_values("r", ascending=False))
rows = []
for t, d in top[["target", "driver"]].itertuples(index=False):
    xy = D[[t, d]].dropna()
    if len(xy) < 48: continue
    with contextlib.redirect_stdout(io.StringIO()): g = grangercausalitytests(xy, maxlag=3)
    p = {k: v[0]["ssr_ftest"][1] for k, v in g.items()}
    rows.append({"target": t, "driver": d, "|r| best": R.loc[t, d], "best lag": L.loc[t, d],
                 "Granger p (min over 1–3)": min(p.values()), "at lag": min(p, key=p.get), "n": len(xy)})
granger = pd.DataFrame(rows).sort_values(["target", "Granger p (min over 1–3)"])
granger.style.format(precision=3).background_gradient(subset=["Granger p (min over 1–3)"], cmap="Greens_r", vmin=0, vmax=0.1)

## 6 · Predictive relevance of the engineered features (h = 3 / 6 / 9)

We use `fe.make_features`, the same leakage-safe builder as notebook 02. The driver list is extended with the candidates found above (zinc, energy PPI, coal, epoxy feedstocks, LME spread and stocks).

For each target and horizon we compute three things:

- **Spearman ρ** between the feature at origin *t* and the forward return `log(y[t+h]/y[t])`;
- **mutual information** (non-linear dependence, `random_state = 0`);
- **regime stability**: the sign of ρ in pre-2021, 2021-22 and 2023+ compared with the full-sample sign.

⚠️ For h > 1 the targets overlap, so successive rows are not independent. Read ρ as a ranking device, not as a significance test.

In [ ]:
X_ext = X.copy()
for c in ["S10", "S11", "CRU_CR_FR", "EPOXY_ECH", "EPOXY_BENZENE"]:
    if c not in X_ext: X_ext[c] = P[c]
EXTRA = ["S05", "S10", "S11", "E13", "E22", "EPOXY_ECH", "EPOXY_BENZENE"]
DRIVERS_STUDY = list(dict.fromkeys(fe.COMMON_DRIVERS + EXTRA))
FEAT = {h: fe.make_features(T, X_ext, h, drivers=DRIVERS_STUDY) for h in HORIZONS}
meta_cols = ["family", "y_level", "y"]
feat_cols = [c for c in FEAT[3].columns if c not in meta_cols]
print(f"{len(feat_cols)} candidate features per (target, horizon); rows per horizon: {len(FEAT[3]):,}")
feat_cols[:12], feat_cols[-8:]

In [ ]:
def score(h: int) -> pd.DataFrame:
    rows = []
    for s, g in FEAT[h].groupby(level="series"):
        g = g.droplevel("series").dropna(subset=["y", "y_level"])
        for f in feat_cols:
            if f.startswith(f"x_{s}_"): continue        # the target's own lag, already in own_*
            xy = g[[f, "y"]].dropna()
            if len(xy) < 36 or xy[f].std() == 0: continue
            rho = xy[f].corr(xy.y, method="spearman")
            mi = mutual_info_regression(xy[[f]].values, xy.y.values, random_state=0, n_neighbors=5)[0]
            signs = []
            for a, b in REGIMES.values():
                sub = xy.loc[a:b]
                if len(sub) >= 12 and sub[f].std() > 0:
                    signs.append(np.sign(sub[f].corr(sub.y, method="spearman")) == np.sign(rho))
            rows.append({"target": s, "horizon": h, "feature": f, "n": len(xy), "rho": rho, "abs_rho": abs(rho),
                         "mi": mi, "stable_regimes": int(sum(signs)), "n_regimes": len(signs)})
    return pd.DataFrame(rows)

SC = pd.concat([score(h) for h in HORIZONS], ignore_index=True)
SC["rank_score"] = SC.groupby(["target", "horizon"])[["abs_rho", "mi"]].rank(pct=True).mean(axis=1)
SC.sort_values("rank_score", ascending=False).groupby(["target", "horizon"]).head(3) \
  .sort_values(["target", "horizon", "rank_score"], ascending=[True, True, False]).head(36)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 9), sharey=True)
order_f = SC.groupby("feature").abs_rho.mean().sort_values(ascending=False).index[:40]
for ax, h in zip(axes, HORIZONS):
    M = SC.query("horizon == @h").pivot(index="feature", columns="target", values="rho").reindex(order_f)
    sns.heatmap(M, cmap=DIV, center=0, vmin=-0.6, vmax=0.6, ax=ax, cbar=(h == 9), cbar_kws={"shrink": .5},
                xticklabels=True, yticklabels=True)
    ax.set_title(f"h = {h}: Spearman ρ(feature_t, return t→t+{h})"); ax.tick_params(labelsize=7); ax.set_xlabel("")
fig.tight_layout(); save(fig, "22_feature_target_rho"); plt.show()

**Reading.**
- **PPIs (S09/S10/S11) are where the signal is.** |ρ| reaches 0.6–0.7 at every horizon. The top features are the **CRU-vs-PPI gap** (`x_gap_steel`, `_dev24`), **own momentum** (`own_dlog_l0`, `own_mom3/6`) and the recent change in **aluminium, coal (E22) and gas (ED02)**. Most of these describe one shared industrial cycle that the smoothed PPIs catch up with. They are not independent drivers.
- **Market prices (CRU, LME, ore) are weak.** |ρ| is 0.15–0.4 and the sign often changes between regimes, which fits the near-random-walk result of notebook 02. The recurring signals are:
  - mean reversion: CRU coil tends to fall when it sits above the PPI (`x_gap_steel` < 0 at h = 6–9), and iron ore reverses its own momentum;
  - a weaker euro over the last 3 months (`x_S06_d3`) comes before higher EUR steel and aluminium prices at h = 9;
  - own volatility: high volatility comes before lower nickel and aluminium returns.
- **Brent → S11 does not show up at h ≥ 3.** It is strong on monthly changes (§5a, r ≈ 0.63 at lag 1). Over 3–9 months, S11's own momentum and the steel/aluminium cycle features rank higher, so Brent is absorbed by them.

## 7 · Multicollinearity: VIF pruning

Each feature also gets a **`core`** flag. The statistical filter alone keeps some cross-family features that are probably spurious (e.g. the plastics PPI "predicting" CRU coil). With about 110 overlapping monthly returns, a coincidental ρ of 0.2–0.3 is easy to find. `core` keeps a selected feature only if it passes one of two tests:
- it has an **economic prior**: an own-series feature, or a driver documented for that target in `fe.DRIVERS`/DATA_OVERVIEW;
- it is sign-stable in **all three** regimes with |ρ| ≥ 0.25.

Candidates per target and horizon are the top-10 features by `rank_score` that pass three filters: **sign-stable in at least 2 of 3 regimes**, |ρ| ≥ 0.10, and at least 60 observations. We then drop collinear features until every VIF is ≤ 5. When several features are above 5, the one with the lowest `rank_score` goes first. With about 110–170 rows per target, a small set is the main protection against the overfitting LightGBM showed in notebook 02.

In [ ]:
TOP_K, MIN_RHO, VIF_MAX = 10, 0.10, 5.0
def vif_prune(df: pd.DataFrame, priority: pd.Series) -> tuple[list[str], pd.Series]:
    cols = list(df.columns)
    while len(cols) > 1:
        Z = df[cols].dropna(); Z = (Z - Z.mean()) / Z.std()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            v = pd.Series([variance_inflation_factor(Z.values, i) for i in range(len(cols))], index=cols)
        if v.max() <= VIF_MAX: return cols, v
        worst = v[v > VIF_MAX]
        cols.remove(worst.index[np.argmin(priority.reindex(worst.index).values)] if len(worst) > 1 else v.idxmax())
    return cols, pd.Series(1.0, index=cols)

SC["candidate"] = False; SC["vif"] = np.nan; SC["selected"] = False
for (s, h), g in SC.groupby(["target", "horizon"]):
    cand = g[(g.stable_regimes >= 2) & (g.n >= 60) & (g.abs_rho >= MIN_RHO)].nlargest(TOP_K, "rank_score")
    SC.loc[cand.index, "candidate"] = True
    data = FEAT[h].xs(s, level="series")[cand.feature.tolist()]
    keep, v = vif_prune(data, cand.set_index("feature").rank_score)
    idx = cand.index[cand.feature.isin(keep)]
    SC.loc[idx, "selected"] = True
    SC.loc[idx, "vif"] = cand.loc[idx, "feature"].map(v).values

# economic prior: own-series features, or drivers documented for this target (fe.DRIVERS, notebook 02 / DATA_OVERVIEW)
PRIOR_EXTRA = {"S11": ["ED01", "E22"], "D02": ["E13"], "S03": ["S06"], "S04": ["S06"]}
def driver_of(f: str) -> str | None:
    if not f.startswith("x_"): return None
    b = f[2:]
    for suf in ("_d1", "_d3", "_dev24"):
        if b.endswith(suf): return b[: -len(suf)]
    return b
SC["driver"] = SC.feature.map(driver_of)
SC["prior"] = SC.feature.str.startswith("own_") | np.array([d in set(fe.DRIVERS.get(t, []) + PRIOR_EXTRA.get(t, []))
                                                   for t, d in zip(SC.target, SC.driver)])
SC["core"] = SC.selected & (SC.prior | ((SC.stable_regimes == 3) & (SC.abs_rho >= 0.25)))

summary = SC.groupby(["target", "horizon"]).agg(candidates=("candidate", "sum"), selected=("selected", "sum"),
                                                core=("core", "sum")).unstack("horizon")
summary

## 8 · Leakage proof

`fe.check_no_leakage` rebuilds the features after scrambling every value after the origin. It asserts that no feature dated ≤ origin changes. We run it for each horizon on three origins (pre-shock, mid-shock and recent) with the **extended** driver list.

In [ ]:
_orig = fe.make_features
fe.make_features = lambda t, x, h, drivers=None: _orig(t, x, h, drivers=drivers or DRIVERS_STUDY)
try:
    for h in HORIZONS:
        for origin in ("2020-06-01", "2022-03-01", "2025-01-01"):
            fe.check_no_leakage(T, X_ext, h, origin)
finally:
    fe.make_features = _orig
print("✓ no leakage: features at every origin are unchanged when the future is scrambled (h = 3, 6, 9)")

## 9 · Selection & export

- **`data/processed/feature_selection.parquet`** has one row per (target, horizon, feature) with ρ, MI, regime stability, VIF, `prior`, `selected` (statistical filter) and `core` (selected + economically justified). Use it to subset features **per target** at training time. Recommended default: `core`; use `selected` as an ablation.
- **`data/processed/features_h{3,6,9}.parquet`** is the leakage-safe design matrix, indexed by (month, series). Columns: the union of the selected features + `family`, `y_level`, `y`. Rows where `y` is NaN are kept, because those are the live forecasting origins; filter on `y.notna()` for training.

In [ ]:
SC.to_parquet(PROC / "feature_selection.parquet", index=False)
for h in HORIZONS:
    cols = SC.query("horizon == @h and selected").feature.unique().tolist()
    out = FEAT[h][meta_cols + sorted(cols)]
    out.to_parquet(PROC / f"features_h{h}.parquet")
    print(f"features_h{h}.parquet: {out.shape[0]:,} rows × {len(cols)} selected features "
          f"({out.y.notna().sum():,} trainable rows)")

sel = SC.query("core").copy()
cnt = sel.groupby(["feature", "horizon"]).size().unstack(fill_value=0)
cnt = cnt.loc[cnt.sum(axis=1).sort_values(ascending=True).index].tail(25)
fig, ax = plt.subplots(figsize=(9, 7))
cnt.plot.barh(stacked=True, ax=ax, color=PAL[:3], width=0.75)
ax.set_title("How many targets keep each feature as `core` (stacked by horizon)"); ax.set_xlabel("# target series")
ax.grid(axis="x", color=GRID); ax.grid(axis="y", visible=False); ax.legend(title="h", loc="lower right")
save(fig, "23_feature_selection"); plt.show()

In [ ]:
FAM = {**fe.FAMILY}
_per_fam = (sel.assign(family=sel.target.map(FAM))
              .groupby(["family", "feature"]).agg(n=("rank_score", "size"), rho=("rho", "mean"))
              .reset_index().sort_values(["family", "n"], ascending=[True, False])
              .groupby("family").head(6))
core_tbl = (SC.query("core").sort_values("rank_score", ascending=False)
              .groupby(["target", "horizon"]).apply(lambda g: ", ".join(f"{f} ({r:+.2f})" for f, r in zip(g.feature, g.rho)))
              .unstack("horizon"))
core_tbl.columns = [f"h={h}: core features (ρ)" for h in core_tbl.columns]
core_tbl

## 10 · Conclusions: features to train on

| Target family | Keep (in business terms) | Evidence in this notebook | Expectation |
|---|---|---|---|
| **EU PPIs** S09 / S10 (steel), S11 (plastics) | The **CRU-vs-PPI gap** (how far the index still has to catch up with the market); own momentum (last month, 3–6 m); the 3-month change in aluminium, coal and gas | ρ 0.6–0.7 with the forward return at h = 3/6/9; CRU CR → S09 at lag 1 = 0.72 | The one family where features clearly help (notebook 02: −30 to −38% error). These are the indices to use in indexation clauses. |
| **Stainless surcharge** SS304 alloy | **Nickel, last month** (the surcharge formula) | CCF 0.71 at lag 1; ρ = 0.42 at h = 3, stable in every window | Strong at h = 3 only; at h = 6–9 nickel itself has to be forecast. |
| **Steel market** CRU CR / HDG, SS304 | Mean reversion vs the PPI (`x_gap_steel`), EUR/USD over 3 months, DE power over 3 months, iron ore / zinc (lag 1) | \|ρ\| 0.2–0.4, sign changes across regimes | Close to a random walk. Keep 2–5 features; expect at most a few % gain over naive. |
| **LME metals** Cu / Al / Ni, iron ore | Own volatility, the LME **futures spread** (vs its 24 m mean), **stocks**, EUR/USD (3 m) | \|ρ\| 0.15–0.35; `x_gap_steel_dev24` also scores (+0.3–0.5), but as a proxy for the industrial cycle | Market-efficient. Spend the effort on calibrated P10/P90 bands, not the point forecast. |
| **Epoxy** (insulating material) | **Benzene** (r 0.66 on monthly changes) and ECH (r 0.38) | Monthly pass-through is weaker than the level-recipe fit (R² 0.98) suggests | Forecast the feedstocks, then derive the resin with the level recipe. |
| **Drop** | 80 redundant series: MEPS copies of CRU, country variants, LME cash/3M duplicates of D01/D02, E01/E02 vs ED01; supplier and benchmark files | \|r\| > 0.95 on monthly changes; or yearly or too sparse | — |

**Caveats.**
- The forward returns overlap for h > 1, and there are only about 110–170 origins per target. A ρ of 0.2–0.3 can be chance. The `core` flag keeps a feature only if it has an economic reason or holds in all three regimes, but the **backtest is the real test**.
- CRU_HDG_DE and S10 are near-duplicates of CRU_CR_DE and S09 (r 0.99 and 0.95). Forecasting them separately adds little information; their forecasts should agree.

**Next step (training).**
1. Load `features_h{h}.parquet` and keep the rows with `y.notna()`.
2. For each target, use the `core` features listed in `feature_selection.parquet`. Run the `selected` set as an ablation.
3. Score both in the same expanding-window backtest as notebook 02 (`src/backtest.py`) against naive.